In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1bCpZCXw6ET9PZPAwiEhKtPdT6lthcWkc", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import torch
import sys
import numpy as np
import matplotlib.pyplot as plt

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs fine on CPU.")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why Memory Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_memory_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# ZeRO Memory Calculator -- Vizuara

In this notebook, you will build a memory calculator that computes the per-GPU memory footprint for each ZeRO stage. By deriving the formulas from first principles and plugging in real numbers, you will develop an intuitive feel for *why* ZeRO is so effective and *when* each stage becomes necessary.

**What you will build:** A complete memory estimation tool that tells you exactly how much GPU memory your model needs under standard data parallelism and each ZeRO stage.

## 1. Why Memory Estimation Matters

Before you start training a large model, you need to answer one question: **will it fit?** If each GPU only has 80 GB of memory (A100) or 24 GB (consumer GPUs), you must know in advance how much memory the model states will consume. Get this wrong, and you waste hours on OOM errors.

The three components of model state memory in mixed-precision training with Adam are:

1. **fp16 weights**: 2 bytes per parameter
2. **fp16 gradients**: 2 bytes per parameter
3. **fp32 optimizer states** (Adam): master weights (4 bytes) + first moment m (4 bytes) + second moment v (4 bytes) = 12 bytes per parameter

Total per parameter: **16 bytes**.

In [ ]:
#@title 🎧 Listen: Math Derivation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_math_derivation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. The Math: Deriving Memory Formulas

Let $\Psi$ be the number of parameters and $N$ the number of GPUs.

| Approach | Weights | Gradients | Optimizer | Total per GPU |
|----------|---------|-----------|-----------|---------------|
| Standard DP | $2\Psi$ | $2\Psi$ | $12\Psi$ | $16\Psi$ |
| ZeRO-1 | $2\Psi$ | $2\Psi$ | $12\Psi / N$ | $4\Psi + 12\Psi/N$ |
| ZeRO-2 | $2\Psi$ | $2\Psi / N$ | $12\Psi / N$ | $2\Psi + 14\Psi/N$ |
| ZeRO-3 | $2\Psi / N$ | $2\Psi / N$ | $12\Psi / N$ | $16\Psi / N$ |

Now let us implement these formulas in code.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Functions Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_memory_functions_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def bytes_to_gb(b):
    """Convert bytes to gigabytes."""
    return b / (1024 ** 3)


def memory_standard_dp(psi):
    """Standard data parallelism: full replication on every GPU."""
    weights = 2 * psi      # fp16 weights
    gradients = 2 * psi    # fp16 gradients
    optimizer = 12 * psi   # fp32 master + m + v
    return {
        'weights_bytes': weights,
        'gradients_bytes': gradients,
        'optimizer_bytes': optimizer,
        'total_bytes': weights + gradients + optimizer,
    }


def memory_zero_stage1(psi, n_gpus):
    """ZeRO Stage 1: partition optimizer states only."""
    weights = 2 * psi
    gradients = 2 * psi
    optimizer = 12 * psi / n_gpus  # partitioned
    return {
        'weights_bytes': weights,
        'gradients_bytes': gradients,
        'optimizer_bytes': optimizer,
        'total_bytes': weights + gradients + optimizer,
    }


def memory_zero_stage2(psi, n_gpus):
    """ZeRO Stage 2: partition optimizer states + gradients."""
    weights = 2 * psi
    gradients = 2 * psi / n_gpus   # partitioned
    optimizer = 12 * psi / n_gpus  # partitioned
    return {
        'weights_bytes': weights,
        'gradients_bytes': gradients,
        'optimizer_bytes': optimizer,
        'total_bytes': weights + gradients + optimizer,
    }


def memory_zero_stage3(psi, n_gpus):
    """ZeRO Stage 3: partition everything."""
    weights = 2 * psi / n_gpus     # partitioned
    gradients = 2 * psi / n_gpus   # partitioned
    optimizer = 12 * psi / n_gpus  # partitioned
    return {
        'weights_bytes': weights,
        'gradients_bytes': gradients,
        'optimizer_bytes': optimizer,
        'total_bytes': weights + gradients + optimizer,
    }


# Quick test with 7B model on 8 GPUs
psi = 7e9
n_gpus = 8

for name, fn in [('Standard DP', lambda: memory_standard_dp(psi)),
                 ('ZeRO-1', lambda: memory_zero_stage1(psi, n_gpus)),
                 ('ZeRO-2', lambda: memory_zero_stage2(psi, n_gpus)),
                 ('ZeRO-3', lambda: memory_zero_stage3(psi, n_gpus))]:
    m = fn()
    print(f"{name:15s}: Weights={bytes_to_gb(m['weights_bytes']):6.1f} GB  "
          f"Grads={bytes_to_gb(m['gradients_bytes']):6.1f} GB  "
          f"Optim={bytes_to_gb(m['optimizer_bytes']):6.1f} GB  "
          f"Total={bytes_to_gb(m['total_bytes']):6.1f} GB")

### Think About This

Look at the numbers above. Standard DP uses ~104 GB per GPU for a 7B model. That exceeds an 80 GB A100. ZeRO-1 brings it to ~36 GB, which fits. ZeRO-3 brings it to ~13 GB.

Now ask yourself: where does the 75% of memory go in ZeRO-1? Answer: the optimizer states (12 bytes per param) are the dominant component, and partitioning them across 8 GPUs reduces them from 78 GB to under 10 GB.

In [ ]:
#@title 🎧 Transition: Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Visualizing Memory Across Stages

Let us create a stacked bar chart to see the memory breakdown visually.

In [ ]:
#@title 🎧 What to Look For: Plot Comparison Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_plot_comparison_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def plot_memory_comparison(psi, n_gpus, model_name="Model"):
    """Create a stacked bar chart comparing memory across ZeRO stages."""
    stages = ['Standard DP', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3']
    fns = [
        lambda: memory_standard_dp(psi),
        lambda: memory_zero_stage1(psi, n_gpus),
        lambda: memory_zero_stage2(psi, n_gpus),
        lambda: memory_zero_stage3(psi, n_gpus),
    ]

    weights_gb = []
    grads_gb = []
    optim_gb = []
    totals = []

    for fn in fns:
        m = fn()
        w = bytes_to_gb(m['weights_bytes'])
        g = bytes_to_gb(m['gradients_bytes'])
        o = bytes_to_gb(m['optimizer_bytes'])
        weights_gb.append(w)
        grads_gb.append(g)
        optim_gb.append(o)
        totals.append(w + g + o)

    x = np.arange(len(stages))
    width = 0.5

    fig, ax = plt.subplots(figsize=(10, 6))

    p1 = ax.bar(x, weights_gb, width, label='Weights (fp16)', color='#2196F3')
    p2 = ax.bar(x, grads_gb, width, bottom=weights_gb, label='Gradients (fp16)', color='#4CAF50')
    p3 = ax.bar(x, optim_gb, width,
                bottom=[w + g for w, g in zip(weights_gb, grads_gb)],
                label='Optimizer States (fp32)', color='#9C27B0')

    # Add total labels on top
    for i, total in enumerate(totals):
        ax.text(i, total + 1, f'{total:.1f} GB', ha='center', va='bottom',
                fontsize=11, fontweight='bold')

    # A100 80 GB line
    ax.axhline(y=80, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
    ax.text(len(stages) - 0.5, 82, 'A100 80 GB', color='red', fontsize=10,
            ha='right', va='bottom')

    ax.set_xlabel('Approach', fontsize=12)
    ax.set_ylabel('Memory per GPU (GB)', fontsize=12)
    psi_b = psi / 1e9
    ax.set_title(f'Memory per GPU: {psi_b:.0f}B {model_name} on {n_gpus} GPUs',
                 fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(stages, fontsize=11)
    ax.legend(fontsize=10, loc='upper right')
    ax.set_ylim(0, max(totals) * 1.15)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


# Visualize for 7B model on 8 GPUs


In [ ]:
#@title 🎧 What to Look For: Plot Comparison Results
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_plot_comparison_results.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
plot_memory_comparison(7e9, 8, model_name="Parameter Model")

In [ ]:
#@title 🎧 Transition: Scaling Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_scaling_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Scaling: How Memory Changes with GPU Count

ZeRO's memory savings scale with the number of GPUs. Let us plot how per-GPU memory decreases as we add more GPUs.

In [ ]:
#@title 🎧 What to Look For: Plot Scaling Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_plot_scaling_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def plot_scaling(psi, gpu_counts):
    """Plot per-GPU memory vs. GPU count for each ZeRO stage."""
    dp_mem = [bytes_to_gb(memory_standard_dp(psi)['total_bytes'])] * len(gpu_counts)
    z1_mem = [bytes_to_gb(memory_zero_stage1(psi, n)['total_bytes']) for n in gpu_counts]
    z2_mem = [bytes_to_gb(memory_zero_stage2(psi, n)['total_bytes']) for n in gpu_counts]
    z3_mem = [bytes_to_gb(memory_zero_stage3(psi, n)['total_bytes']) for n in gpu_counts]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(gpu_counts, dp_mem, 'o-', label='Standard DP', color='#F44336', linewidth=2)
    ax.plot(gpu_counts, z1_mem, 's-', label='ZeRO-1', color='#FF9800', linewidth=2)
    ax.plot(gpu_counts, z2_mem, '^-', label='ZeRO-2', color='#4CAF50', linewidth=2)
    ax.plot(gpu_counts, z3_mem, 'D-', label='ZeRO-3', color='#2196F3', linewidth=2)

    ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(gpu_counts[-1], 82, 'A100 80 GB', color='red', fontsize=9, ha='right')

    ax.set_xlabel('Number of GPUs', fontsize=12)
    ax.set_ylabel('Memory per GPU (GB)', fontsize=12)
    psi_b = psi / 1e9
    ax.set_title(f'Per-GPU Memory Scaling: {psi_b:.0f}B Parameters', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xticks(gpu_counts)
    plt.tight_layout()
    plt.show()


In [ ]:
#@title 🎧 What to Look For: Plot Scaling Results
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_plot_scaling_results.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
gpu_counts = [1, 2, 4, 8, 16, 32, 64]
plot_scaling(7e9, gpu_counts)

### Think About This

Notice how Standard DP is a flat line at 104 GB -- adding GPUs does not reduce per-GPU memory at all. ZeRO-1 and ZeRO-2 flatten out quickly because the non-partitioned components (weights, or weights + gradients) set a floor. Only ZeRO-3 continues to decrease toward zero because *everything* is partitioned.

In [ ]:
#@title 🎧 Listen: Comm Cost Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_comm_cost_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Communication Cost Analysis

Memory savings are great, but what about communication overhead? Let us quantify the communication volume for each stage.

- **Standard DP / ZeRO-1 / ZeRO-2**: AllReduce or equivalent = $4\Psi$ bytes per GPU
- **ZeRO-3**: $6\Psi$ bytes per GPU (1.5x standard DP)

The 1.5x comes from the extra AllGather of weights during the forward pass.

In [ ]:
#@title 🎧 Code Walkthrough: Comm Volume Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_comm_volume_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def communication_volume(psi, stage):
    """Communication volume per GPU in bytes for one training step."""
    if stage in ['dp', 'zero1', 'zero2']:
        # AllReduce (or reduce-scatter + allgather) for gradients
        return 4 * psi  # 2 * 2*psi for the allreduce
    elif stage == 'zero3':
        # Forward allgather + backward allgather + reduce-scatter
        return 6 * psi  # 3 * 2*psi
    else:
        raise ValueError(f"Unknown stage: {stage}")


psi = 7e9

stages_info = [
    ('Standard DP', 'dp'),
    ('ZeRO-1', 'zero1'),
    ('ZeRO-2', 'zero2'),
    ('ZeRO-3', 'zero3'),
]

print(f"Communication volume per GPU per step ({psi/1e9:.0f}B params):")
print(f"{'Stage':15s} {'Volume (GB)':>12s} {'vs DP':>8s}")
print("-" * 38)

dp_vol = communication_volume(psi, 'dp')
for name, stage in stages_info:
    vol = communication_volume(psi, stage)
    ratio = vol / dp_vol
    print(f"{name:15s} {bytes_to_gb(vol):10.1f} GB  {ratio:6.2f}x")

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Your Turn -- TODO Exercises

### TODO 1: Complete the Memory-Communication Trade-off Table

Build a function that produces a comprehensive comparison table for any model size and GPU count. It should show per-GPU memory, total cluster memory, redundancy factor, and communication overhead.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def zero_comparison_table(psi, n_gpus):
    """
    Build a comparison table for all ZeRO stages.

    Returns a list of dicts, each with keys:
      'stage', 'mem_per_gpu_gb', 'total_cluster_gb',
      'redundancy_factor', 'comm_vs_dp'
    """
    results = []

    # ============ TODO ============
    # For each stage (DP, ZeRO-1, ZeRO-2, ZeRO-3), compute:
    # 1. mem_per_gpu_gb: bytes_to_gb of the per-GPU total
    # 2. total_cluster_gb: mem_per_gpu_gb * n_gpus
    # 3. redundancy_factor: total_cluster_gb / unique_data_gb
    #    where unique_data_gb = bytes_to_gb(16 * psi)
    # 4. comm_vs_dp: ratio of comm volume to standard DP
    #
    # Hint: Use the memory_* and communication_volume functions above

    unique_gb = bytes_to_gb(16 * psi)

    for stage_name, mem_fn, comm_stage in [
        ('Standard DP', lambda: memory_standard_dp(psi), 'dp'),
        ('ZeRO-1', lambda: memory_zero_stage1(psi, n_gpus), 'zero1'),
        ('ZeRO-2', lambda: memory_zero_stage2(psi, n_gpus), 'zero2'),
        ('ZeRO-3', lambda: memory_zero_stage3(psi, n_gpus), 'zero3'),
    ]:
        mem = mem_fn()
        per_gpu = bytes_to_gb(mem['total_bytes'])
        total_cluster = ???    # YOUR CODE: per_gpu * n_gpus
        redundancy = ???       # YOUR CODE: total_cluster / unique_gb
        comm_ratio = ???       # YOUR CODE: communication_volume ratio

        results.append({
            'stage': stage_name,
            'mem_per_gpu_gb': per_gpu,
            'total_cluster_gb': total_cluster,
            'redundancy_factor': redundancy,
            'comm_vs_dp': comm_ratio,
        })
    # ==============================

    return results


# Test it
table = zero_comparison_table(7e9, 8)
print(f"{'Stage':15s} {'Per GPU':>10s} {'Cluster':>10s} {'Redundancy':>12s} {'Comm':>8s}")
print("-" * 58)
for row in table:
    print(f"{row['stage']:15s} {row['mem_per_gpu_gb']:8.1f} GB"
          f" {row['total_cluster_gb']:8.1f} GB"
          f" {row['redundancy_factor']:10.2f}x"
          f" {row['comm_vs_dp']:6.2f}x")

In [ ]:
#@title 🎧 Before You Start: Todo1 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_todo1_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 1
table = zero_comparison_table(7e9, 8)
assert len(table) == 4, f"Expected 4 rows, got {len(table)}"
assert abs(table[0]['redundancy_factor'] - 8.0) < 0.01, (
    f"Standard DP redundancy should be 8.0x with 8 GPUs, got {table[0]['redundancy_factor']:.2f}")
assert abs(table[3]['comm_vs_dp'] - 1.5) < 0.01, (
    f"ZeRO-3 comm should be 1.5x, got {table[3]['comm_vs_dp']:.2f}")
print("TODO 1 passed!")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Find the Minimum GPUs Needed

Given a model size and GPU memory capacity, determine the minimum number of GPUs required to fit the model under each ZeRO stage. This is one of the most practical questions in distributed training.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def min_gpus_needed(psi, gpu_memory_gb, activation_overhead_gb=0):
    """
    Find the minimum number of GPUs to fit the model for each ZeRO stage.

    Args:
        psi: number of parameters
        gpu_memory_gb: available GPU memory in GB
        activation_overhead_gb: estimated activation memory (we subtract this
                                from the available memory for model states)

    Returns:
        dict mapping stage name to minimum GPU count (or 'N/A' if impossible)
    """
    available_gb = gpu_memory_gb - activation_overhead_gb
    results = {}

    # ============ TODO ============
    # For each stage, find the smallest N (from 1 to 1024) such that
    # the per-GPU memory (in GB) fits within available_gb.
    #
    # Standard DP: memory is constant regardless of N (16*psi per GPU)
    #   - If 16*psi fits, N=1 works. Otherwise, 'N/A'.
    # ZeRO-1: formula is 4*psi + 12*psi/N bytes
    #   - Solve for N: try N=1,2,...,1024
    # ZeRO-2: formula is 2*psi + 14*psi/N bytes
    # ZeRO-3: formula is 16*psi/N bytes

    stage_fns = {
        'Standard DP': memory_standard_dp,
        'ZeRO-1': memory_zero_stage1,
        'ZeRO-2': memory_zero_stage2,
        'ZeRO-3': memory_zero_stage3,
    }

    for stage_name, fn in stage_fns.items():
        found = 'N/A'
        for n in range(1, 1025):
            if stage_name == 'Standard DP':
                mem = fn(psi)
            else:
                mem = ???  # YOUR CODE: call fn with psi and n
            if bytes_to_gb(mem['total_bytes']) <= available_gb:
                found = ???  # YOUR CODE: what should found be?
                break
        results[stage_name] = found
    # ==============================

    return results


# Test with a 30B model on A100-80GB with 10 GB for activations
result = min_gpus_needed(30e9, gpu_memory_gb=80, activation_overhead_gb=10)
print("Minimum GPUs for 30B model (A100-80GB, 10 GB activation overhead):")
for stage, n in result.items():
    print(f"  {stage}: {n} GPUs")

In [ ]:
#@title 🎧 Before You Start: Todo2 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo2_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 2
result = min_gpus_needed(30e9, gpu_memory_gb=80, activation_overhead_gb=10)
assert result['Standard DP'] == 'N/A', "Standard DP should not fit a 30B model on 80GB GPU"
assert isinstance(result['ZeRO-3'], int) and result['ZeRO-3'] <= 16, (
    f"ZeRO-3 should fit 30B on fewer than 16 GPUs, got {result['ZeRO-3']}")
print("TODO 2 passed!")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Add Activation Memory Estimation

Model states are only part of the picture. Activations (saved during the forward pass for backpropagation) also consume significant memory. For a Transformer model:

$$A \approx s \cdot b \cdot h \cdot L \cdot (10 + \frac{24 \cdot s}{h})$$

where $s$ = sequence length, $b$ = micro-batch size, $h$ = hidden dimension, $L$ = number of layers. This is an approximation from the Megatron-LM paper.

In [ ]:
#@title 🎧 Before You Start: Todo3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_todo3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def estimate_activation_memory(seq_len, batch_size, hidden_dim, num_layers):
    """
    Estimate activation memory for a Transformer model (bytes per GPU).

    Uses the Megatron-LM approximation:
        A = s * b * h * L * (10 + 24*s/h)

    Args:
        seq_len: sequence length (s)
        batch_size: micro-batch size (b)
        hidden_dim: hidden dimension (h)
        num_layers: number of transformer layers (L)

    Returns:
        activation memory in bytes
    """
    # ============ TODO ============
    # Implement the formula above.
    # Return the value in bytes.

    activation_bytes = ???  # YOUR CODE HERE

    # ==============================
    return activation_bytes


# Test: LLaMA-7B config: h=4096, L=32, s=2048, b=1
act_bytes = estimate_activation_memory(
    seq_len=2048, batch_size=1, hidden_dim=4096, num_layers=32
)
print(f"Estimated activation memory for LLaMA-7B (s=2048, b=1):")
print(f"  {bytes_to_gb(act_bytes):.1f} GB")
print(f"\nCompare this to model states: ~13 GB (ZeRO-3 on 8 GPUs)")
print(f"Activations can be a significant fraction of total memory!")

In [ ]:
#@title 🎧 Before You Start: Todo3 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_todo3_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 3
act_bytes = estimate_activation_memory(2048, 1, 4096, 32)
act_gb = bytes_to_gb(act_bytes)
assert 3 < act_gb < 30, f"Activation estimate seems off: {act_gb:.1f} GB (expected 5-25 GB)"
print(f"Activation memory: {act_gb:.1f} GB -- TODO 3 passed!")

In [ ]:
#@title 🎧 Transition: Planner Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_planner_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Complete Memory Planner

Let us build a comprehensive planner that combines model states + activations to give a complete picture.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Planner Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_memory_planner_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def memory_planner(model_config, n_gpus, gpu_memory_gb=80):
    """
    Complete memory planner for distributed training.

    Args:
        model_config: dict with keys 'params', 'hidden_dim', 'num_layers',
                      'seq_len', 'micro_batch_size'
        n_gpus: number of GPUs
        gpu_memory_gb: per-GPU memory capacity
    """
    psi = model_config['params']
    psi_b = psi / 1e9

    # Activation memory
    act_bytes = estimate_activation_memory(
        model_config['seq_len'],
        model_config['micro_batch_size'],
        model_config['hidden_dim'],
        model_config['num_layers'],
    )
    act_gb = bytes_to_gb(act_bytes)

    print(f"=" * 65)
    print(f"  Memory Planner: {psi_b:.0f}B Model on {n_gpus} x {gpu_memory_gb} GB GPUs")
    print(f"=" * 65)
    print(f"  Sequence length: {model_config['seq_len']}")
    print(f"  Micro-batch size: {model_config['micro_batch_size']}")
    print(f"  Estimated activation memory: {act_gb:.1f} GB\n")

    available = gpu_memory_gb - act_gb
    print(f"  Available for model states: {available:.1f} GB\n")

    print(f"  {'Stage':15s} {'Model States':>14s} {'+ Activations':>14s} {'Fits?':>8s}")
    print(f"  " + "-" * 55)

    for name, fn in [('Standard DP', lambda: memory_standard_dp(psi)),
                     ('ZeRO-1', lambda: memory_zero_stage1(psi, n_gpus)),
                     ('ZeRO-2', lambda: memory_zero_stage2(psi, n_gpus)),
                     ('ZeRO-3', lambda: memory_zero_stage3(psi, n_gpus))]:
        m = fn()
        states_gb = bytes_to_gb(m['total_bytes'])
        total = states_gb + act_gb
        fits = "YES" if total <= gpu_memory_gb else "NO"
        marker = "  " if fits == "YES" else "XX"
        print(f"  {name:15s} {states_gb:12.1f} GB {total:12.1f} GB  {marker} {fits}")

    print()


# LLaMA-7B
memory_planner({
    'params': 7e9, 'hidden_dim': 4096, 'num_layers': 32,
    'seq_len': 2048, 'micro_batch_size': 1
}, n_gpus=8, gpu_memory_gb=80)

# LLaMA-70B
memory_planner({
    'params': 70e9, 'hidden_dim': 8192, 'num_layers': 80,
    'seq_len': 2048, 'micro_batch_size': 1
}, n_gpus=64, gpu_memory_gb=80)

In [ ]:
#@title 🎧 Listen: Real Memory Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_real_memory_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Measuring Real Memory with PyTorch

The formulas above are theoretical. Let us verify them by actually creating model parameters and measuring GPU memory usage.

In [ ]:
#@title 🎧 Code Walkthrough: Real Memory Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_real_memory_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Create a model with known parameter count
    # 100M parameters so it fits on any GPU
    psi_test = 100_000_000  # 100M

    # Theoretical prediction
    theoretical_gb = bytes_to_gb(16 * psi_test)

    # Create fp16 weights
    weights_fp16 = torch.randn(psi_test, dtype=torch.float16, device='cuda')
    mem_after_weights = torch.cuda.memory_allocated() / 1e9

    # Create fp16 gradients (simulated)
    grads_fp16 = torch.randn(psi_test, dtype=torch.float16, device='cuda')
    mem_after_grads = torch.cuda.memory_allocated() / 1e9

    # Create fp32 optimizer states (master weights + m + v)
    master_weights = weights_fp16.float()  # 4 bytes per param
    moment_m = torch.zeros(psi_test, dtype=torch.float32, device='cuda')
    moment_v = torch.zeros(psi_test, dtype=torch.float32, device='cuda')
    mem_after_optim = torch.cuda.memory_allocated() / 1e9

    print(f"Parameter count: {psi_test/1e6:.0f}M")
    print(f"Theoretical total: {theoretical_gb:.3f} GB")
    print(f"")
    print(f"After fp16 weights:    {mem_after_weights:.3f} GB  (theory: {bytes_to_gb(2*psi_test):.3f} GB)")
    print(f"After fp16 gradients:  {mem_after_grads:.3f} GB  (theory: {bytes_to_gb(4*psi_test):.3f} GB)")
    print(f"After fp32 optimizer:  {mem_after_optim:.3f} GB  (theory: {bytes_to_gb(16*psi_test):.3f} GB)")
    print(f"")
    print(f"Match: {abs(mem_after_optim - theoretical_gb) / theoretical_gb * 100:.1f}% difference")

    # Clean up
    del weights_fp16, grads_fp16, master_weights, moment_m, moment_v
    torch.cuda.empty_cache()
else:
    print("No GPU available. Skipping real memory measurement.")
    print("The theoretical formulas above still apply.")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### Think About This
1. Why does ZeRO-1 alone give a 2.9x memory reduction for a 7B model on 8 GPUs? What happens to this ratio with more GPUs?
2. At what model size does even ZeRO-3 on 8 GPUs become insufficient (assuming 80 GB GPUs)?
3. Why does ZeRO-3 have 1.5x the communication of standard DP, but ZeRO-1 and ZeRO-2 have the same? What changes?

### What Comes Next
In Notebook 2, we will implement a simplified version of ZeRO-1 and ZeRO-2 from scratch in pure PyTorch. You will see exactly how optimizer state partitioning and reduce-scatter work at the code level.

### Key Takeaway
ZeRO eliminates memory redundancy in data parallelism by partitioning model states across GPUs. The three stages offer a clear trade-off: more partitioning means less memory per GPU, with ZeRO-3 being the most aggressive (1/N memory) at the cost of 1.5x communication overhead.